# DE séquentiel (CPU) — Colab

Ce notebook compile et exécute le **DE séquentiel** du projet « Massively Parallel PSO and DE algorithms » :

```text
main.cpp
kernel.h
kernel.cpp
kernel.cu
```

Ce prototype est volontairement **CPU uniquement** (`run_de_sequential`, pas de kernel CUDA) : l'objectif est de valider la logique DE/rand/1/bin (mutation, crossover, sélection) avant le portage GPU (Phase 6, plus tard). `kernel.cu` reste le PSO fourni **intact** — il est compilé pour garder les 4 fichiers cohérents avec `nvcc`, mais n'est pas utilisé par ce chemin d'exécution.

Aucun GPU n'est strictement nécessaire pour ce notebook, mais on garde les vérifications GPU/nvcc pour rester prêt pour les phases suivantes.

## 0. (Optionnel) Activer un GPU Colab

Ce notebook fonctionne sans GPU. Si tu veux garder un runtime GPU prêt pour la suite du projet : **Exécution → Modifier le type d'exécution → Accélérateur matériel → GPU (T4)**.

In [ ]:
import os
import shutil
import subprocess
import zipfile
import re
from pathlib import Path

print('=== Vérification du GPU (optionnel) ===')
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout if result.stdout else 'Pas de GPU détecté (OK pour ce notebook, le DE séquentiel tourne sur CPU).')

## 1. Vérifier `nvcc`

On utilise `nvcc` pour compiler les 4 fichiers ensemble (comme demandé dans le sujet, §9-10), même si `kernel.cu` n'est pas exécuté par ce prototype. Les images GPU Colab embarquent déjà un toolkit CUDA complet : **pas de réinstallation manuelle de CUDA/drivers**.

In [ ]:
print('=== Recherche de nvcc ===')
nvcc = shutil.which('nvcc')

if not nvcc:
    candidates = sorted(Path('/usr/local').glob('cuda*/bin/nvcc'), reverse=True)
    for candidate in candidates:
        if candidate.exists():
            cuda_bin = str(candidate.parent)
            os.environ['PATH'] = cuda_bin + ':' + os.environ['PATH']
            nvcc = shutil.which('nvcc')
            if nvcc:
                print('✅ nvcc trouvé après ajout au PATH :', nvcc)
                break

if nvcc:
    print('✅ nvcc :', nvcc)
    subprocess.run([nvcc, '--version'], check=False)
else:
    raise RuntimeError(
        "❌ nvcc introuvable. Active un runtime GPU (Exécution → Modifier le type d'exécution → GPU) "
        "puis relance cette cellule — nvcc est nécessaire même pour compiler du code host-only ici."
    )

## 2. Dossier du projet et upload des 4 fichiers

Sélectionne les 4 fichiers du DE séquentiel :

- `main.cpp`
- `kernel.h`
- `kernel.cpp`
- `kernel.cu`

(un `.zip` du projet fonctionne aussi — extraction automatique à l'étape suivante).

In [ ]:
PROJECT_DIR = Path('/content/DE_CUDA')
PROJECT_DIR.mkdir(parents=True, exist_ok=True)

from google.colab import files
uploaded = files.upload()

for name in uploaded.keys():
    src = Path('/content') / name
    dst = PROJECT_DIR / name
    shutil.move(str(src), str(dst))
    print('Copié :', dst)

In [ ]:
zip_files = list(PROJECT_DIR.glob('*.zip'))
for z in zip_files:
    print('Extraction de', z.name)
    with zipfile.ZipFile(z, 'r') as archive:
        archive.extractall(PROJECT_DIR)
    z.unlink()

required = ['main.cpp', 'kernel.h', 'kernel.cpp', 'kernel.cu']
for filename in required:
    target = PROJECT_DIR / filename
    if not target.exists():
        matches = list(PROJECT_DIR.rglob(filename))
        if matches:
            shutil.copy2(matches[0], target)
            print(f'{filename}: trouvé dans {matches[0]}')

print('\n=== Vérification des fichiers ===')
missing = []
for filename in required:
    path = PROJECT_DIR / filename
    if path.exists():
        print('✅', filename, f'({path.stat().st_size} octets)')
    else:
        print('❌', filename, 'MANQUANT')
        missing.append(filename)

if missing:
    raise FileNotFoundError('Fichiers manquants : ' + ', '.join(missing))

## 3. Compilation

`main.cpp` + `kernel.cpp` + `kernel.cu` compilés ensemble avec `nvcc` (les 4 fichiers restent cohérents pour la suite du projet). Pas de `-lcurand` : le code actuel (PSO comme DE séquentiel) ne l'utilise pas.

In [ ]:
os.chdir(PROJECT_DIR)

command = [
    nvcc,
    '-O2',
    '-std=c++17',
    '-diag-suppress=546',   # transfer of control bypasses initialization (switch objectif dans kernel.cu)
    '-o', 'programde',
    'main.cpp',
    'kernel.cpp',
    'kernel.cu',
]

print('Commande :')
print(' '.join(command))

compile_result = subprocess.run(command, capture_output=True, text=True)
print('\n--- stdout ---')
print(compile_result.stdout)
print('\n--- stderr ---')
print(compile_result.stderr)

if compile_result.returncode != 0:
    raise RuntimeError('❌ Compilation échouée (voir stderr ci-dessus).')

print('✅ Compilation réussie :', PROJECT_DIR / 'programde')

## 4. Vérifier l'exécutable et son usage

In [ ]:
exe = PROJECT_DIR / 'programde'
print('Executable :', exe, '—', exe.stat().st_size, 'octets')
!/content/DE_CUDA/programde   # sans arguments : affiche l'usage attendu

## 5. Lancer un test et parser le résultat

`run_de(dim, pop, objectif, seed)` appelle le binaire, vérifie le code retour et **parse la ligne de sortie** (`DE-CPU  time  minimum  FEs`) en dictionnaire — pratique pour construire un tableau de résultats ensuite.

Rappel des contraintes du binaire : `dim > 0`, **`population >= 4`** (DE/rand/1 a besoin de r1,r2,r3 distincts et différents de i), `objectif` dans `[0,4]` (0=Levy 1=Rastrigin 2=Rosenbrock 3=Griewank 4=Sphere).

In [ ]:
OUTPUT_RE = re.compile(
    r'DE-CPU\s+([\d.]+)\s+(-?[\d.eE+-]+)\s+(\d+)'
)

def run_de(dim, population, objective=1, seed=42, timeout=300):
    if population < 4:
        raise ValueError('population doit être >= 4 (contrainte DE/rand/1).')

    cmd = [str(exe), str(dim), str(population), str(objective), str(seed)]
    print('Commande :', ' '.join(cmd))
    try:
        result = subprocess.run(
            cmd, cwd=PROJECT_DIR, capture_output=True, text=True, timeout=timeout
        )
    except subprocess.TimeoutExpired:
        raise RuntimeError(f'⏱️ Timeout ({timeout}s) dépassé pour dim={dim}, pop={population}')

    if result.stdout:
        print(result.stdout.strip())
    if result.stderr:
        print('--- stderr ---')
        print(result.stderr)
    if result.returncode != 0:
        raise RuntimeError(
            f'Programme terminé avec le code {result.returncode} '
            f'(dim={dim}, pop={population}, objectif={objective}, seed={seed})'
        )

    match = OUTPUT_RE.search(result.stdout)
    if not match:
        raise RuntimeError(f'Sortie inattendue, impossible de parser :\n{result.stdout}')

    return {
        'dimension': dim,
        'population': population,
        'objective': objective,
        'seed': seed,
        'time_sec': float(match.group(1)),
        'minimum': float(match.group(2)),
        'fes': int(match.group(3)),
    }

run_de(10, 50, 1, 42)

## 6. Validation progressive (§11 du sujet)

On monte en charge dans l'ordre demandé : D=10/NP=50 → D=50/NP=100 → D=100/NP=500, sur la fonction Rastrigin (objectif=1) pour commencer.

In [ ]:
run_de(10, 50, 1, 42)
run_de(50, 100, 1, 42)
run_de(100, 500, 1, 42)

## 7. Tester les 4 fonctions demandées par le sujet

Rastrigin (1), Rosenbrock (2), Griewank (3), Sphère (4) — Levy (0) est aussi dans le code mais ne fait pas partie des 4 fonctions demandées en §14.

⚠️ Griewank (objectif=3) a un bug hérité du code fourni : le produit de cosinus est neutralisé (accumulateur initialisé à 0 au lieu de 1), donc le minimum observé sera décalé de +1 par rapport à l'optimum théorique. Pas corrigé pour l'instant — voir discussion précédente.

In [ ]:
OBJ_NAMES = {0: 'Levy', 1: 'Rastrigin', 2: 'Rosenbrock', 3: 'Griewank', 4: 'Sphere'}

for objective in [1, 2, 3, 4]:
    print(f'\n### {OBJ_NAMES[objective]} (objectif={objective}) ###')
    run_de(10, 50, objective, 42)

## 8. Matrice complète des expériences

Dimensions × populations × les 4 fonctions demandées, une seule graine pour l'instant (les 10 runs moyenne/écart-type viendront en Phase 13, une fois l'architecture GPU validée). Résultats collectés dans un `DataFrame` pandas.

In [ ]:
import pandas as pd

dimensions = [10, 50, 100]
populations = [50, 100, 500]
objectives = [1, 2, 3, 4]
seed = 42

results = []
for dim in dimensions:
    for pop in populations:
        for objective in objectives:
            print(f'\n=== D={dim}, NP={pop}, {OBJ_NAMES[objective]} ===')
            results.append(run_de(dim, pop, objective, seed))

print('\n✅ Toutes les configurations ont été exécutées.')
df_results = pd.DataFrame(results)
df_results['objective_name'] = df_results['objective'].map(OBJ_NAMES)
df_results

## 9. Exporter les résultats

In [ ]:
csv_path = PROJECT_DIR / 'resultats_de_sequentiel.csv'
df_results.to_csv(csv_path, index=False)
print('✅ Résultats sauvegardés :', csv_path)

files.download(str(csv_path))